In [ ]:
# !cp /kaggle/input/datasets/ivanadolfoahumada/funciones-code-bert/ /kaggle/working/functions -r
# !rm -rf /kaggle/working/functions

# Primera prueba

Pasos a seguir según lo que se me ocurre

## 1 - Exploración manual

- Analizar los labels y contenido del csv
- Analizar estadísticas varias del dataset (cantidad de entradas, tamaño de input, tamaño post-tokenización sin preprocesamiento)
- Analizar balance de clases
- Buscar como se recolectaron los datos

## 2 - Data cleaning

- Balancear de ser necesario
- Aceptar vulnerabilidades según si un score ($score_c$), definido por la suma de la capacidad de cada herramienta para detectar el problema normalizada, sobrepasa un umbral ($\mu_c$) (**hiperparámetro**), que sirve para aceptar o no labels y entradas. (**matriz de capacidad de detección**, **MCD**, filas son las herramientas $i$, columnas son las categorias $c$.) Si ningún label, ni siquiera clean, sobrepasa el umbral, se descarta.

$$score_c = \frac{\sum_{i \in \text{detectaron } c} e^{\, rate_{i,c} / (100 \cdot T)}}{n_{tools}}$$

- $score_c$: valor a comparar contra el umbral de aceptación de un label para un contrato.
- $rate_{i,c}$: acc de la herramienta i, para la categoría c.
- $T$: Temperatura, a mayor T, mayor es la distancia entre la peor herramienta para esa categoria y la mejor. **Hiperparámetro**.

(Se necesita un análisis más profundo sobre falsos positivos y negativos, además de establecer pruebas para evaluar si esto mejora o empeora el rendimiento general del modelo. Smartbugs incluye herramientas dentro de las 9 tools, asi que capaz sería mejor sacarlo.)

- Elegir un tipo de representación del código
- (1) Filtrar por tamaño (2) Truncar inputs que superan los 512 tokens
- Data merging, de-duplication
- Normalización
- Volver a evaluar estadísticas

## 3 - Entrenamiento

- Según las últimas estadísticas evaluadas evaluar que tecnicas extra usar
- Definir los parámetros de regularizadores
- Definir un batch size que mantenga el entrenamiento por debajo de 30 min
- Por esta etapa dejar adamW con los valores base que viene asi que no hay que tocar el optimizador

## 4 - Optimización

- Probar Optuna
- Diseñar estudios para optimización de learning rate y regularizadores
- (1) quasi-random search (2) baysian optimization (3) grid search

## 5 - Evaluar resultados

- Graficar todo lo graficable

## 6 - IDEAS

- Matriz de capacidad de detección.
- LCL Line CLassification Token, resumen de la linea (que tan involucrada está en la vulnerabilidad.)
- FCL Function CLassification Token, resúmen de la función (que tan involucrada está en la vulnerabilidad.)
- Dos ramas, errores locales y globales (estos pasapalabra.)
- RMT Recurrent Memory Transfer.
- Post procesamiento y presentación de resultados procesados por Qwen 7b o mistral 1b (que interpreten las cosas y muestren la audit.)
- Agarrar Smartbugs y MANDO-HGT para que evalúen el dataset.

## Trabajo que falta

- Investigar data cleaning.

Los datos fueron recolectados haciendo scrapping en etherscan, mientras que las vulnerabilidades se detectaron usando las herramientas que se muestran en tools, algunas pueden ser poco accurate y es lo que hay, se puede usar smartbugs wild curated para comprobar pero son re pocos contratos, después toca probar, que.se.yo.

Usaron google Big Query, o sea, herramienta de Google para hacer Big Queries.

Eliminaron duplicados, elijo creerles.

In [ ]:
import pandas as pd

df = pd.read_csv("/kaggle/input/datasets/tranduongminhdai/smartbug-dataset/smartbugs_wild.csv")

print(df.columns.tolist())

In [ ]:
import ast
from pprint import pprint 

a = 5
print(df["address"][a], '\n')
#pprint(ast.literal_eval(df["tools"][a]), indent=4)
print(df["lines"][a])
# print(df["source_code"][a])

Las columnas del dataset son las siguientes:
- `address`: Dirección del contrato en la blockchain ethereum.
- `tools`: Diccionario que tiene cada herramienta con dos diccionarios adentro, `categories` (categorias de vuls que detectó y cantidad) y `vulnerabilities`(vulnerabilidades específicas y cantidad.)
- `lines`: Ni la más mínima idea.
- `nb_vulnerabilities`: Cantidad total de vulnerabilidades.
- `source_code`: Código fuente en solidity así como está.

Antes de analizar cosas hay que poner el dataset en un estado que tenga sentido analizar.

# 2 - Data cleaning

## Tareas
- Normalizar (sacar \t, \n y \r)
- (1) sacar todos los comentarios (2) dejar los comentarios.
- Calcular hash MD5 y comparar para eliminar duplicados
- Buscar si los contratos de Smartbugs Curated están en Wild y removerlos para evitar Data Leakage.
- Usando Mando-hgt, agregarlos a tools con el mismo formato que las otras.
- Evaluar Mango-hgt con smartbugs-curated.
- Aceptar vulnerabilidades según si un score ($score_c$), definido por la suma de la capacidad de cada herramienta para detectar el problema normalizada, sobrepasa un umbral ($\mu_c$) (**hiperparámetro**), que sirve para aceptar o no labels y entradas. (**matriz de capacidad de detección**, **MCD**, filas son las herramientas $i$, columnas son las categorias $c$.) Si ningún label, ni siquiera clean, sobrepasa el umbral, se descarta.

$$score_c = \frac{\sum_{i \in \text{detectaron } c} e^{\, rate_{i,c} / (100 \cdot T)}}{n_{tools}}$$

- $score_c$: valor a comparar contra el umbral de aceptación de un label para un contrato.
- $rate_{i,c}$: acc de la herramienta i, para la categoría c.
- $T$: Temperatura, a mayor T, mayor es la distancia entre la peor herramienta para esa categoria y la mejor. **Hiperparámetro**.

(Se necesita un análisis más profundo sobre falsos positivos y negativos, además de establecer pruebas para evaluar si esto mejora o empeora el rendimiento general del modelo. Smartbugs incluye herramientas dentro de las 9 tools, asi que capaz sería mejor sacarlo.)

- Tokenizar todo y (1) descartar lo que sea mayor a 512 tokens (2) Truncar o descartar según umbral X.

## Validación
- Evaluar balance de clases.
- Evaluar cantidad total de entradas.
- Usando resultados de entrenamiento después vemos.

In [ ]:
def extract_labels(tools_dict):
    labels = set()
    for tool, result in tools_dict.items():
        # Extraigo categories porque es un problema más sencillo que las vulnerabilidades en sí
        for vuln in result.get('categories', {}).keys():
            labels.add(vuln.lower())
    return labels

In [ ]:
import ast

# Convierte a un dict a los valores de tools que son strings que representan dicts en la columna tools
df['tools'] = df['tools'].apply(ast.literal_eval)

# Extraigo categorias de vulnerabilidades de cada tools elem
df['vulnerability_category_labels'] = df['tools'].apply(extract_labels)

# Obtengo la lista de labels
labels = set()

for category_set in df['vulnerability_category_labels']:
    if not category_set:
        labels.add('clean')
    else:
        for category in category_set:
            labels.add(category)

print('Labels: ', labels)

In [ ]:
label_dict_count = dict.fromkeys(labels, 0)

for category_set in df['vulnerability_category_labels']:
    if not category_set:
        label_dict_count['clean'] += 1
    else:
        for category in category_set:
            label_dict_count[category] += 1

for label in label_dict_count:
    print(label.ljust(20), '\t\t', label_dict_count[label])

In [ ]:
n_before = len(df)
df = df[df['source_code'].apply(lambda x: isinstance(x, str) and len(x) > 0)].reset_index(drop=True)
print(f'Contratos con source_code inválido removidos: {n_before - len(df)}')
print(f'Contratos restantes                         : {len(df)}')

In [ ]:
import sys
sys.path.append('/kaggle/working/')

from functions.preprocessing import preprocess_source

df['source_code'] = df['source_code'].apply(preprocess_source)
print('Normalización y eliminación de comentarios completada.')
print(f"Ejemplo (primeros 300 chars):\n{df['source_code'].iloc[0][:300]}")

In [ ]:
from functions.deduplication import deduplicate_df

df = deduplicate_df(df)

In [ ]:
from functions.data_leakage import check_curated_leakage

# Reemplazar el path cuando el dataset esté disponible
df = check_curated_leakage(df, curated_path='/kaggle/input/<PATH>/smartbugs_curated.csv')

In [ ]:
# TODO: Mando-HGT — agregar resultados al dict de tools con el mismo formato
# que las demás herramientas una vez que esté disponible.
#
# Formato esperado por contrato:
# tools_dict['mando_hgt'] = {
#     'categories': {'reentrancy': 2},
#     'vulnerabilities': {'Reentrancy': 2}
# }
#
# Evaluar Mando-HGT con Smartbugs Curated y agregar su fila a functions/mcd.py.

print('Mando-HGT: PENDIENTE.')

In [ ]:
from functions.mcd import apply_mcd_filter, T, MU_C, EQUAL_RATE, EQUAL_WEIGHT_CATS

print(f'Hiperparámetros: T={T}, μ_c={MU_C}, EQUAL_RATE={EQUAL_RATE}')
print(f'Categorías con peso uniforme: {EQUAL_WEIGHT_CATS}')
print()
df = apply_mcd_filter(df, t=1.2, mu_c=0.3, equal_rate=20.0)

In [ ]:
from transformers import RobertaTokenizer
from functions.tokenization import filter_by_token_length, truncate_by_token_length

tokenizer = RobertaTokenizer.from_pretrained('microsoft/codebert-base')
df = truncate_by_token_length(df, tokenizer, max_tokens=512)

In [ ]:
from functions.stats import print_class_balance

print_class_balance(df)

In [ ]:
import ast



df1 = pd.read_csv('/kaggle/working/discard_plus512_t12_mu03.csv')
df1['accepted_labels'] = df1['accepted_labels'].apply(ast.literal_eval)
print_class_balance(df1, 'accepted_labels')

df2 = pd.read_csv('/kaggle/working/truncate_plus512_t1_mu05.csv')
df2['accepted_labels'] = df2['accepted_labels'].apply(ast.literal_eval)
print_class_balance(df2)

In [ ]:
df.to_csv('truncate_plus512_t1.2_mu03_bis.csv')

# 3 - Entrenamiento

## Tareas
- TBD

## Validación
- F1, Acc, Recall, probar VDS.
- Gráficos, muchos gráficos.

In [ ]:
import os
os.environ['HF_TOKEN'] = 'TOKEN'
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"  # Disable HF transfer
os.environ['TRANSFORMERS_CACHE'] = os.path.join(os.getcwd(), "cache")
os.environ['HF_HOME'] = os.path.join(os.getcwd(), "cache")

In [ ]:
"""
Fine-tuning CodeBERT on SmartBugs dataset for smart contract vulnerability detection.
Multi-label classification (train) / Single-label evaluation (test).
Environment: Kaggle notebook with 2x GPU T4
"""
 
import os
import ast
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import f1_score, recall_score, accuracy_score
from tqdm import tqdm
 
# Config

tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base")
model = AutoModel.from_pretrained("microsoft/codebert-base")

# Para usar las dos GPU
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

model = model.to(DEVICE)

TRAIN_PATH = "/kaggle/input/datasets/ivanadolfoahumada/codebert-train/discard_plus512_t12_mu03.csv"
TEST_PATH  = "/kaggle/input/datasets/ivanadolfoahumada/smartbugs-curated-test/smartbugs_curated_test.csv"
MODEL_NAME = "/kaggle/input/models/jsday96/codebert-base/transformers/default/1/codebert-base"
MAX_LEN    = 512
BATCH_SIZE = 8
EPOCHS     = 5
LR         = 2e-5
WARMUP_RATIO = 0.1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


# carga y preproc

train_df = pd.read_csv(TRAIN_PATH)
test_df  = pd.read_csv(TEST_PATH)
 
def parse_labels(label_str):
    """Convert '{label1, label2}' string to a sorted list of labels."""
    label_str = label_str.strip()
    if label_str.startswith("{") and label_str.endswith("}"):
        inner = label_str[1:-1]
        return sorted([l.strip().strip("'\"") for l in inner.split(",") if l.strip()])
    try:
        parsed = ast.literal_eval(label_str)
        if isinstance(parsed, (set, list, tuple)):
            return sorted(str(x) for x in parsed)
        return [str(parsed)]
    except Exception:
        return [label_str.strip()]
 
train_df["labels_list"] = train_df["accepted_labels"].apply(parse_labels)
test_df["labels_list"]  = test_df["vulnerability_category"].apply(lambda x: [x.strip()])
 
# Fit binarizer on union of all labels
mlb = MultiLabelBinarizer()
mlb.fit(train_df["labels_list"].tolist() + test_df["labels_list"].tolist())
NUM_LABELS = len(mlb.classes_)
print(f"Classes ({NUM_LABELS}): {list(mlb.classes_)}")
 
train_encoded = mlb.transform(train_df["labels_list"])
test_encoded  = mlb.transform(test_df["labels_list"])
 
# 2. Dataset & DataLoader


class SmartBugsDataset(Dataset):
    def __init__(self, sources, labels, tokenizer, max_len):
        self.sources   = sources.reset_index(drop=True)
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len
 
    def __len__(self):
        return len(self.sources)
 
    def __getitem__(self, idx):
        code = str(self.sources[idx])
        enc  = self.tokenizer(
            code,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.float),
        }
 
train_dataset = SmartBugsDataset(train_df["source_code"], train_encoded, tokenizer, MAX_LEN)
test_dataset  = SmartBugsDataset(test_df["source_code"],  test_encoded,  tokenizer, MAX_LEN)
 
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2)


# Model — CodeBERT + classification head

class CodeBERTClassifier(nn.Module):
    def __init__(self, model_name, num_labels, dropout=0.3):
        super().__init__()
        self.encoder = model
        self.dropout    = nn.Dropout(dropout)
        self.classifier = nn.Linear(768, num_labels)
 
    def forward(self, input_ids, attention_mask):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = self.dropout(out.last_hidden_state[:, 0, :])   # [CLS]
        return self.classifier(cls)
 
model = CodeBERTClassifier(MODEL_NAME, NUM_LABELS).to(DEVICE)

optimizer   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler   = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(total_steps * WARMUP_RATIO),
    num_training_steps=total_steps,
)
criterion = nn.BCEWithLogitsLoss()
 
# Entrenamiento

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}")
 
    for batch in pbar:
        ids   = batch["input_ids"].to(DEVICE)
        mask  = batch["attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
 
        logits = model(ids, mask)
        loss   = criterion(logits, labels)
 
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
 
        total_loss += loss.item()
        pbar.set_postfix(loss=f"{loss.item():.4f}")
 
    print(f"  -> Epoch {epoch+1} avg loss: {total_loss / len(train_loader):.4f}")
 
# Res

model.eval()
all_logits, all_labels = [], []
 
with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        ids  = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
 
        logits = model(ids, mask).cpu().numpy()
        all_logits.append(logits)
        all_labels.append(batch["labels"].numpy())
 
all_logits = np.vstack(all_logits)
all_labels = np.vstack(all_labels)
 
# Predicciones

all_preds = (torch.sigmoid(torch.tensor(all_logits)).numpy() >= 0.5).astype(int)
for i in range(len(all_preds)):
    if all_preds[i].sum() == 0:
        all_preds[i][np.argmax(all_logits[i])] = 1
 
# Test is single-label → flatten for standard metrics
pred_flat = np.argmax(all_preds,  axis=1)
true_flat = np.argmax(all_labels, axis=1)
 
f1_macro = f1_score(true_flat, pred_flat, average="macro",  zero_division=0)
f1_micro = f1_score(true_flat, pred_flat, average="micro",  zero_division=0)
recall_m = recall_score(true_flat, pred_flat, average="macro", zero_division=0)
acc      = accuracy_score(true_flat, pred_flat)
 
print("\n" + "=" * 50)
print("TEST RESULTS")
print("=" * 50)
print(f"  F1 Macro : {f1_macro:.4f}")
print(f"  F1 Micro : {f1_micro:.4f}")
print(f"  Recall   : {recall_m:.4f}")
print(f"  Accuracy : {acc:.4f}")
print("=" * 50)
 
# ---------------------------------------------------------------------------
# 7. Save
# ---------------------------------------------------------------------------
out_dir = "/kaggle/working/codebert_smartbugs"
os.makedirs(out_dir, exist_ok=True)
state = model.module.state_dict() if hasattr(model, "module") else model.state_dict()
torch.save(state, os.path.join(out_dir, "model.pt"))
tokenizer.save_pretrained(out_dir)
print(f"\nModel saved to {out_dir}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Using 2 GPUs
Using device: cuda
Classes (9): ['access_control', 'arithmetic', 'clean', 'denial_service', 'front_running', 'other', 'reentrancy', 'time_manipulation', 'unchecked_low_calls']


Epoch 1/5: 100%|██████████| 637/637 [05:48<00:00,  1.83it/s, loss=0.2815]


  -> Epoch 1 avg loss: 0.3705


Epoch 2/5:  49%|████▉     | 313/637 [02:51<02:57,  1.83it/s, loss=0.1722]